# Nemotron v7.5 — Training Notebook

Trains plain rank-32 LoRA on the **`data/processed/all_categorical_splits/`** dataset (10,545 records, 100% solver-verified).

**Dataset (9 per-category JSONL files):**
| Category | Records | Solver Acc | Notes |
|---|---|---|---|
| bit_manipulation        | 2,728 | 100% | augmented |
| cipher                  | 1,576 | 100% | Tong baseline |
| cryptarithm_deduce      | 659   | 100% | Z3-verified |
| cryptarithm_guess       | 164   | 100% | Z3-verified |
| equation_numeric_deduce | 540   | 100% | filtered |
| equation_numeric_guess  | 111   | 100% | 81.6% optimized |
| gravity                 | 1,597 | 100% | Tong baseline |
| numeral                 | 1,576 | 100% | Tong baseline |
| unit_conversion         | 1,594 | 100% | Tong baseline |
| **TOTAL**               | **10,545** | **100%** | — |

**Differences from v7.4:**
- Data source switched to per-category splits (10,545 vs 9,500 records, 100% vs 87.7% solver-correct)
- Stratified batching uses the **explicit `category` field** in each JSONL (no keyword inference)
- Plain LoRA only — **NO** DoRA / rsLoRA / PiSSA / LoRA+ (those broke at vLLM inference, see v7.2 regression)
- Same install path & imports as v7 → **dependencies untouched**

**Eval-server contract (vLLM):**
- max_lora_rank = 32, max_tokens = 7680, temperature = 0.0 (greedy → dropout MUST be 0), max_model_len = 8192.

In [ ]:
# ============================================================
# 1. OFFLINE DEPENDENCY INSTALLATION  (identical to v7 — DO NOT change)
# ============================================================
import subprocess, sys, os
from pathlib import Path

def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read().strip()
            rel_pack_path = pth_file.parent / relpath
            if rel_pack_path.exists():
                sys.path.append(str(rel_pack_path))

offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
target_dir  = "/kaggle/working/packages"
os.makedirs(target_dir, exist_ok=True)

# Resolve both hyphen and underscore mount-name variants Kaggle has used
resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/")
resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/")

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index", "--find-links", offline_dir,
        "--target", target_dir,
        "datasets", "trl", "peft"
    ])
    print("Offline packages installed.")

# wandb — must be set BEFORE import so it never tries to connect
os.environ["WANDB_MODE"] = "offline"

WANDB_AVAILABLE = False
try:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index", "--find-links", offline_dir,
        "--target", target_dir,
        "wandb"
    ])
    WANDB_AVAILABLE = True
    print("wandb installed (offline).")
except Exception:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wandb"])
        WANDB_AVAILABLE = True
        print("wandb installed (online).")
    except Exception:
        print("wandb not available — training will continue without W&B logging.")

sys.path.append(target_dir)
resolve_python_path(target_dir)

In [ ]:
# ============================================================
# 2. IMPORTS & ENVIRONMENT  (identical to v7 — DO NOT change)
# ============================================================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import stat, shutil, zipfile, time, json, re, glob
import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm

if WANDB_AVAILABLE:
    import wandb

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"W&B     : {'offline mode' if WANDB_AVAILABLE else 'disabled'}")

In [ ]:
# ============================================================
# 2b. WEIGHTS & BIASES — OFFLINE MODE
# ============================================================
WANDB_PROJECT  = "nemotron-v75"
WANDB_RUN_NAME = "v75-lora-r32-a64-allcat-100pct"
WANDB_DIR      = "/kaggle/working"

if WANDB_AVAILABLE:
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        dir=WANDB_DIR,
        config={
            "model": "Nemotron-3-Nano-30B-A3B",
            "data": "all_categorical_splits (9 files, 10,545 records, 100% verified)",
            "lora_rank": 32,
            "lora_alpha": 64,
            "learning_rate": 1e-4,
            "num_epochs": 3,
            "batch_size": 4,
            "grad_accum": 8,
            "effective_batch": 32,
            "max_seq_len": 4096,
            "lora_targets": "q,k,v,o,in,out,up,down,lm_head",
            "lora_dropout": 0.0,
            "warmup_steps": 50,
            "scheduler": "cosine",
            "packing": False,
            "stratified": True,
            "bf16": True,
            "adapter_kind": "plain LoRA (no DoRA/rsLoRA/PiSSA/LoRA+)",
        },
        tags=["nemotron", "lora", "v75", "sft", "all_categorical_splits"],
    )
    print(f"W&B offline run initialized: {wandb.run.dir}")
else:
    print("W&B not available — training metrics logged to stdout only.")

In [ ]:
# ============================================================
# 3. TRITON FIXES (rmsnorm + ptxas-blackwell binary copy) — DEFENSIVE
# ============================================================
# (a) rmsnorm_fn replaced with pure PyTorch (mamba_ssm Triton kernel
#     fails on some GPU configs)
# (b) ptxas-blackwell binary lives in a read-only mount without +x;
#     copy to /tmp + chmod, redirect Triton env vars, bust caches

def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    var = x.pow(2).mean(-1, keepdim=True)
    y = x * torch.rsqrt(var + eps)
    out = y * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None:    out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, "rmsnorm_fn"):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# Find ptxas-blackwell — handle both hyphen and underscore mount-paths
candidates = (
    glob.glob("/kaggle/usr/lib/notebooks/**/ptxas-blackwell", recursive=True)
    + glob.glob("/kaggle/usr/lib/notebooks/**/ptxas", recursive=True)
    + glob.glob("/usr/local/cuda*/bin/ptxas", recursive=True)
    + glob.glob("/usr/local/lib/python*/dist-packages/nvidia/cuda_nvcc/bin/ptxas",
                recursive=True)
)
src = next((c for c in candidates if "blackwell" in c), None) \
   or (candidates[0] if candidates else None)

if src and os.path.exists(src):
    dst = "/tmp/ptxas-blackwell"
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    for v in ("TRITON_PTXAS_PATH", "TRITON_PTXAS_BLACKWELL_PATH",
              "TRITON_PTXAS_BIN", "TRITON_PTXAS"):
        os.environ[v] = dst

    try:
        import triton.backends.nvidia.compiler as nv_compiler
        try: nv_compiler.get_ptxas_version.cache_clear()
        except AttributeError: pass
        nv_compiler.get_ptxas_version = lambda arch: "release 12.8"
        from triton import knobs as triton_knobs
        for attr in ("ptxas", "ptxas_blackwell"):
            triton_knobs.nvidia.__dict__.pop(attr, None)
    except Exception as e:
        print(f"Triton cache clear warning: {e}")

    print(f"[ok] ptxas binary -> {dst}  (copied from {src})")
else:
    print("[warn] no ptxas binary found anywhere — Mamba kernel will crash")

In [ ]:
# ============================================================
# 4. HYPERPARAMETERS — v7.5 (plain LoRA, eval-server safe)
# ============================================================
# Eval-server contract (vLLM):
#   max_lora_rank          = 32        -> r <= 32
#   max_tokens             = 7680
#   max_model_len          = 8192
#   temperature            = 0.0       -> greedy (dropout MUST be 0)
#   gpu_memory_utilization = 0.85
#
# Why plain LoRA only:
#   DoRA/rsLoRA/PiSSA modify training-time math, but vLLM applies the
#   STANDARD formula  delta = (alpha/r) * B*A  at inference. The math
#   mismatch caused the v7.2 regression (0.69 -> 0.58). Sticking to
#   plain LoRA keeps train-time math == inference-time math.
#
# Max CoT length in this dataset (verified):
#   bit_manipulation max ≈ 2,553 tokens  (rich)
#   cipher max         ≈ 3,302 tokens  (max overall)
#   equation_deduce    ≈ 3,120 tokens
# So MAX_SEQ_LEN = 4096 leaves comfortable headroom (no record dropped).

LORA_RANK           = 32          # eval-server cap
LORA_ALPHA          = 64          # 2:1 ratio
MAX_SEQ_LEN         = 4096        # CoT max ≈ 3,302 tokens; 4096 is safe
NUM_EPOCHS          = 3           # LoRA sweet spot
BATCH_SIZE          = 4           # per-device
GRAD_ACCUM          = 8           # effective batch = 32
LR                  = 1e-4        # standard for plain LoRA r=32
WARMUP_STEPS        = 50
SAVE_EVERY_N_EPOCHS = 1

USE_STRATIFIED_BATCHING = True    # one category per effective batch

MODEL_PATH  = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
OUTPUT_DIR  = "/kaggle/working/adapter"
CKPT_DIR    = "/kaggle/working/checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

# 9 per-category JSONL files — try multiple Kaggle dataset mount points.
CATEGORY_FILES = [
    "train_cot_bit_manipulation.jsonl",
    "train_cot_cipher.jsonl",
    "train_cot_cryptarithm_deduce.jsonl",
    "train_cot_cryptarithm_guess.jsonl",
    "train_cot_equation_numeric_deduce.jsonl",
    "train_cot_equation_numeric_guess.jsonl",
    "train_cot_gravity.jsonl",
    "train_cot_numeral.jsonl",
    "train_cot_unit_conversion.jsonl",
]

DATA_DIR_CANDIDATES = [
    "/kaggle/input/nemotron-categorical-splits",
    "/kaggle/input/nemotron-categorical-splits/all_categorical_splits",
    "/kaggle/input/all-categorical-splits",
    "/kaggle/input/all-categorical-splits/all_categorical_splits",
    "/kaggle/input/nemotron-cot-categorical/all_categorical_splits",
    "/kaggle/input/nemotron-cot-categorical",
    # Local-machine fallback for dry runs
    str(Path.cwd().parent / "data" / "processed" / "all_categorical_splits"),
    str(Path.cwd() / "data" / "processed" / "all_categorical_splits"),
]

# Eval-server assertions
assert LORA_RANK   <= 32,   f"LORA_RANK={LORA_RANK} exceeds eval cap 32"
assert MAX_SEQ_LEN <= 7680, f"MAX_SEQ_LEN={MAX_SEQ_LEN} exceeds eval cap 7680"

print(f"Epochs     : {NUM_EPOCHS}  (checkpoint every {SAVE_EVERY_N_EPOCHS})")
print(f"LR         : {LR:.1e}  (plain LoRA, no rsLoRA scaling)")
print(f"Batch      : {BATCH_SIZE}×{GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} eff  |  rank={LORA_RANK}, alpha={LORA_ALPHA}")
print(f"Max seqlen : {MAX_SEQ_LEN}")
print(f"Stratified : {USE_STRATIFIED_BATCHING}")
print(f"Adapter    : plain LoRA (no DoRA/rsLoRA/PiSSA/LoRA+)")
print(f"Ckpt dir   : {CKPT_DIR}")

In [ ]:
# ============================================================
# 5. CALLBACKS — progress bar + per-epoch checkpoint zip
# ============================================================

class LiveProgressCallback(TrainerCallback):
    """Live tqdm bar with loss + ETA."""
    def __init__(self):
        self.pbar       = None
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(total=state.max_steps, desc="Training",
                         unit="step", dynamic_ncols=True, file=sys.stdout)
        self.start_time = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar is None:
            return
        elapsed = time.time() - self.start_time
        step    = state.global_step
        eta     = (elapsed / step) * (state.max_steps - step) if step > 0 else 0
        loss_str = (f"loss={state.log_history[-1]['loss']:.4f}"
                    if state.log_history and "loss" in state.log_history[-1] else "loss=...")
        self.pbar.set_postfix_str(f"{loss_str}  elapsed={elapsed/60:.1f}m  eta={eta/60:.1f}m")
        self.pbar.update(1)
        sys.stdout.flush()

    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.close()


class CheckpointZipCallback(TrainerCallback):
    """After every N epochs: save adapter -> patch config -> zip to CKPT_DIR."""

    def __init__(self, ckpt_dir, output_dir, every_n=1):
        self.ckpt_dir   = ckpt_dir
        self.output_dir = output_dir
        self.every_n    = every_n
        self.epoch_losses = {}

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = round(state.epoch)
        if epoch % self.every_n != 0:
            return

        # 1. Save adapter weights (plain LoRA — no PiSSA conversion needed)
        epoch_dir = os.path.join(self.output_dir, f"epoch_{epoch:02d}")
        os.makedirs(epoch_dir, exist_ok=True)
        model.save_pretrained(epoch_dir)

        # 2. Patch base_model_name_or_path to canonical Kaggle name
        cfg_path = os.path.join(epoch_dir, "adapter_config.json")
        with open(cfg_path) as f:
            cfg = json.load(f)
        cfg["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
        with open(cfg_path, "w") as f:
            json.dump(cfg, f, indent=2)

        # 3. Zip ALL files in epoch_dir
        zip_name = f"adapter_epoch_{epoch:02d}.zip"
        zip_path = os.path.join(self.ckpt_dir, zip_name)
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for fname in sorted(os.listdir(epoch_dir)):
                fp = os.path.join(epoch_dir, fname)
                if os.path.isfile(fp):
                    zf.write(fp, arcname=fname)

        # 4. Record loss
        recent_losses = [h["loss"] for h in state.log_history if "loss" in h]
        avg_loss = sum(recent_losses[-10:]) / len(recent_losses[-10:]) if recent_losses else float("nan")
        self.epoch_losses[epoch] = avg_loss

        zip_mb = os.path.getsize(zip_path) / 1024 / 1024
        print(f"\n[Epoch {epoch:02d}] ✓ {zip_name}  ({zip_mb:.1f} MB)  avg_loss={avg_loss:.4f}")

        if WANDB_AVAILABLE and wandb.run is not None:
            wandb.log({"epoch_checkpoint/epoch": epoch,
                       "epoch_checkpoint/avg_loss": avg_loss,
                       "epoch_checkpoint/zip_mb": zip_mb}, step=state.global_step)

    def print_summary(self):
        if not self.epoch_losses:
            return
        print("\n  Checkpoint loss summary:")
        best_epoch = min(self.epoch_losses, key=self.epoch_losses.get)
        for ep, loss in sorted(self.epoch_losses.items()):
            marker = " ← best" if ep == best_epoch else ""
            print(f"    epoch {ep:02d}: loss={loss:.4f}{marker}")
        print(f"\n  Best checkpoint: adapter_epoch_{best_epoch:02d}.zip  (use for submission)")


ckpt_callback = CheckpointZipCallback(
    ckpt_dir=CKPT_DIR, output_dir=OUTPUT_DIR, every_n=SAVE_EVERY_N_EPOCHS
)
print("Callbacks ready: LiveProgressCallback + CheckpointZipCallback")

In [ ]:
# ============================================================
# 6. LOAD DATA — 9 per-category JSONL files from all_categorical_splits/
# ============================================================
# Each record already has {"category": "...", "messages": [...]} structure,
# so we use the explicit category field (no keyword inference needed).

data_dir = None
for cand in DATA_DIR_CANDIDATES:
    if cand and os.path.isdir(cand):
        # Verify at least one of the expected files is present
        if any(os.path.exists(os.path.join(cand, f)) for f in CATEGORY_FILES):
            data_dir = cand
            break

if data_dir is None:
    raise FileNotFoundError(
        "all_categorical_splits/ directory not found.\n"
        "Upload the 9 JSONL files as a Kaggle dataset and add it as input,\n"
        "e.g. dataset name 'nemotron-categorical-splits'.\n"
        f"Searched: {DATA_DIR_CANDIDATES}"
    )

print(f"Found data directory: {data_dir}\n")

all_records = []
per_cat_counts = {}
for fname in CATEGORY_FILES:
    fpath = os.path.join(data_dir, fname)
    if not os.path.exists(fpath):
        print(f"  [skip] {fname} not present")
        continue
    n = 0
    with open(fpath) as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            # Ensure category field exists (fall back to filename-derived)
            if "category" not in rec or not rec["category"]:
                rec["category"] = fname.replace("train_cot_", "").replace(".jsonl", "")
            all_records.append(rec)
            n += 1
    per_cat_counts[fname] = n
    print(f"  loaded {n:>5} from {fname}")

print(f"\nTOTAL records loaded: {len(all_records)}")

In [ ]:
# ============================================================
# 7. TOKENIZE & FORMAT — apply chat template, keep category labels
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

all_texts  = []
all_labels = []
fallback_template = 0

for rec in all_records:
    msgs = [m for m in rec["messages"] if m["role"] != "system"]
    try:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
    except Exception:
        fallback_template += 1
        text = (
            f"<|im_start|>user\n{msgs[0]['content']}<|im_end|>\n"
            f"<|im_start|>assistant\n{msgs[-1]['content']}<|im_end|>"
        )
    all_texts.append(text)
    all_labels.append(rec["category"])

if fallback_template:
    print(f"[warn] Used fallback Chat-ML template for {fallback_template} records")

hf_dataset = Dataset.from_dict({"text": all_texts, "label": all_labels})
print(f"\nFormatted dataset: {len(hf_dataset)} examples\n")

from collections import Counter
dist = Counter(all_labels)
print("Category distribution:")
for name, n in dist.most_common():
    print(f"  {name:30s} {n:5d}  ({100*n/len(all_labels):.1f}%)")

print("\nSample (first 400 chars):")
print(hf_dataset[0]['text'][:400])

In [ ]:
# ============================================================
# 8. DROP OVERSIZED SAMPLES (> MAX_SEQ_LEN tokens)
# ============================================================
# Verified data max ≈ 3,302 tokens; MAX_SEQ_LEN = 4096 should drop 0.
print(f"Filtering samples > {MAX_SEQ_LEN} tokens...")
before = len(hf_dataset)

def get_token_length(example):
    ids = tokenizer(example['text'], truncation=False,
                    return_attention_mask=False)['input_ids']
    return {'token_len': len(ids)}

hf_dataset = hf_dataset.map(get_token_length, desc="Counting tokens")
hf_dataset = hf_dataset.filter(
    lambda x: x['token_len'] <= MAX_SEQ_LEN,
    desc="Dropping oversized",
)
hf_dataset = hf_dataset.remove_columns(['token_len'])
print(f"  Kept {len(hf_dataset)} / {before}  ({before - len(hf_dataset)} dropped)")

steps_estimate = len(hf_dataset) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS
print(f"\nEstimated steps : {steps_estimate}")
print(f"Estimated time  : ~{steps_estimate * 8 / 3600:.1f} hrs (rough; ~8s/step on H100)")

In [ ]:
# ============================================================
# 9. LOAD MODEL — bf16, NO quantization
# ============================================================
# NemotronH = HYBRID (52 layers): 23 Attention + 23 Mamba-2 + 6 MoE.
# Does NOT support flash_attention_2 — use default eager attention.

flash_whl = "/kaggle/input/nvidia-nemotron-offline-packages/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
if os.path.exists(flash_whl):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-index", flash_whl])
        print("Installed flash_attn wheel (used by internal kernels)")
    except Exception as e:
        print(f"flash_attn install skipped: {e}")

print("Loading base model (bf16, eager attention)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map={"": 0},
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.gradient_checkpointing_enable()

# Disable Mamba fast path on NemotronH
for name, mod in list(sys.modules.items()):
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False

print("Model loaded on GPU")

In [ ]:
# ============================================================
# 10. APPLY LoRA — plain rank-32 (eval-server safe, no exotic flags)
# ============================================================
# delta = (alpha/r) * B*A   <- vLLM applies this exact formula.
# Train-time math == inference-time math. No surprises.

LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",   # Attention  (23 layers)
    "in_proj", "out_proj",                      # Mamba-2 SSM (23 layers)
    "up_proj", "down_proj",                     # MLP / MoE experts
    "lm_head",                                  # Output head
]

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    modules_to_save=["embed_tokens"],
    lora_dropout=0.0,                    # greedy decoding -> must be 0
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    # NO use_dora, NO use_rslora, NO init_lora_weights="pissa..."
    # All those train-time tricks break vLLM's plain-LoRA inference math.
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Triton compiler fix
try:
    import triton.backends.nvidia.compiler as nv_compiler
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
    nv_compiler.get_ptxas_version = lambda arch: "12.0"
except Exception as e:
    print(f"Triton compiler fix skipped: {e}")

In [ ]:
# ============================================================
# 11. TRAINING — Stratified SFT, one category per effective batch
# ============================================================
# Each effective batch (32 samples) holds ONE category, so gradients are
# coherent (no contradictory signals from mixing bit-ops + cipher + ...).
# Round-robin across categories so every domain is seen frequently.

from torch.utils.data import Sampler
import random as _random

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM


def build_stratified_index_order(labels, chunk_size, seed=0):
    """Return an index list where each `chunk_size`-block is one label."""
    buckets = {}
    for i, lbl in enumerate(labels):
        buckets.setdefault(lbl, []).append(i)

    rng = _random.Random(seed)
    for lbl in buckets:
        rng.shuffle(buckets[lbl])

    order = []
    active = list(buckets.keys())
    rng.shuffle(active)
    while active:
        next_active = []
        for lbl in active:
            take = buckets[lbl][:chunk_size]
            buckets[lbl] = buckets[lbl][chunk_size:]
            order.extend(take)
            if buckets[lbl]:
                next_active.append(lbl)
        active = next_active
    return order


class PrecomputedOrderSampler(Sampler):
    def __init__(self, labels, chunk_size, num_epochs, base_seed=1337):
        self.labels      = list(labels)
        self.chunk_size  = chunk_size
        self.num_epochs  = num_epochs
        self.base_seed   = base_seed
        self.epoch       = 0
        self._current_order = build_stratified_index_order(
            self.labels, chunk_size, seed=base_seed
        )

    def set_epoch(self, epoch):
        self.epoch = epoch
        self._current_order = build_stratified_index_order(
            self.labels, self.chunk_size, seed=self.base_seed + epoch
        )

    def __iter__(self):
        return iter(self._current_order)

    def __len__(self):
        return len(self.labels)


class StratifiedSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_labels=None, chunk_size=16,
                 num_epochs=3, **kwargs):
        self._strat_labels = stratified_labels
        self._strat_chunk  = chunk_size
        self._strat_epochs = num_epochs
        super().__init__(*args, **kwargs)

    def _get_train_sampler(self, *args, **kwargs):
        if self._strat_labels is None:
            return super()._get_train_sampler(*args, **kwargs)
        return PrecomputedOrderSampler(
            labels=self._strat_labels,
            chunk_size=self._strat_chunk,
            num_epochs=self._strat_epochs,
        )


# -------- SFTConfig (NO loraplus_lr_ratio — TRL <0.11 breaks on it) --------
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch_fused",
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    save_strategy="no",                  # CheckpointZipCallback handles saving
    report_to="wandb" if WANDB_AVAILABLE else "none",
    run_name=WANDB_RUN_NAME if WANDB_AVAILABLE else None,
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,                       # packing breaks per-sample label alignment
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=2,
    remove_unused_columns=False,         # keep 'label' column for sampler
)

labels_for_sampler = list(hf_dataset['label']) if USE_STRATIFIED_BATCHING else None

if USE_STRATIFIED_BATCHING:
    print(f"Using STRATIFIED batching (chunk = {EFFECTIVE_BATCH} samples/category)")
    trainer = StratifiedSFTTrainer(
        model=model,
        train_dataset=hf_dataset,
        processing_class=tokenizer,
        args=training_args,
        callbacks=[LiveProgressCallback(), ckpt_callback],
        stratified_labels=labels_for_sampler,
        chunk_size=EFFECTIVE_BATCH,
        num_epochs=NUM_EPOCHS,
    )
else:
    print("Using standard RANDOM batching")
    trainer = SFTTrainer(
        model=model,
        train_dataset=hf_dataset,
        processing_class=tokenizer,
        args=training_args,
        callbacks=[LiveProgressCallback(), ckpt_callback],
    )

steps_per_epoch = trainer.state.max_steps // NUM_EPOCHS if hasattr(trainer.state, 'max_steps') else "?"
print(f"\nStarting training:")
print(f"  {len(hf_dataset)} samples  |  {NUM_EPOCHS} epochs  |  max_seq_len={MAX_SEQ_LEN}")
print(f"  LR={LR:.1e}  warmup={WARMUP_STEPS} steps")
print(f"  batch={BATCH_SIZE}×{GRAD_ACCUM}={EFFECTIVE_BATCH} eff  |  steps/epoch≈{steps_per_epoch}")
print(f"  Stratified: {USE_STRATIFIED_BATCHING}  |  Checkpoint every {SAVE_EVERY_N_EPOCHS} epoch(s) → {CKPT_DIR}")
print(f"  W&B: {'offline logging' if WANDB_AVAILABLE else 'disabled'}\n")

t0 = time.time()
trainer.train()
elapsed_hrs = (time.time() - t0) / 3600
print(f"\nTraining complete! Time: {elapsed_hrs:.2f} hrs")

ckpt_callback.print_summary()

In [ ]:
# ============================================================
# 12. SAVE FINAL ADAPTER — plain LoRA (no PiSSA conversion needed)
# ============================================================
trainer.model.save_pretrained(OUTPUT_DIR)

# Patch base_model_name_or_path to canonical Kaggle model name
config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
with open(config_path) as f:
    adapter_config = json.load(f)

adapter_config["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"

with open(config_path, "w") as f:
    json.dump(adapter_config, f, indent=2)

print(f"base_model_name_or_path -> {adapter_config['base_model_name_or_path']}")
print(f"peft_type              -> {adapter_config.get('peft_type')}")
print(f"r / alpha              -> {adapter_config.get('r')} / {adapter_config.get('lora_alpha')}")
print(f"use_dora               -> {adapter_config.get('use_dora', False)}  (must be False/absent)")
print(f"use_rslora             -> {adapter_config.get('use_rslora', False)}  (must be False/absent)")

# Verify weights look trained
try:
    from safetensors import safe_open
    with safe_open(os.path.join(OUTPUT_DIR, "adapter_model.safetensors"),
                   framework="pt") as f:
        keys  = list(f.keys())
        norms = [f.get_tensor(k).norm().item() for k in keys[:5]]
    print(f"\nAdapter tensors: {len(keys)} parameters")
    print(f"First 5 weight norms: {[f'{n:.4f}' for n in norms]}")
    if all(n < 0.001 for n in norms):
        print("WARNING: Norms near 0 — adapter may be untrained!")
    else:
        print("Adapter looks healthy (non-zero weights).")
except Exception as e:
    print(f"Could not verify safetensors: {e}")

print(f"\nFiles in {OUTPUT_DIR}:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {fname}  ({size_mb:.2f} MB)")

In [ ]:
# ============================================================
# 13. ZIP FINAL ADAPTER + LIST PER-EPOCH CHECKPOINTS
# ============================================================
ZIP_PATH = "/kaggle/working/adapter.zip"
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

file_count = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, arcname=fname)
            file_count += 1

with zipfile.ZipFile(ZIP_PATH) as zf:
    contents = zf.namelist()

zip_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024

print("=" * 60)
print(f"  FINAL adapter.zip  ({zip_mb:.1f} MB)  — {file_count} files")
print(f"  Contents: {contents}")
print("=" * 60)

ckpt_zips = sorted(
    [f for f in os.listdir(CKPT_DIR) if f.endswith(".zip")],
    key=lambda x: int(x.replace("adapter_epoch_", "").replace(".zip", ""))
    if x.replace("adapter_epoch_", "").replace(".zip", "").isdigit() else 0
)

print(f"\nPer-epoch checkpoints in {CKPT_DIR}:")
total_ckpt_mb = 0
for zname in ckpt_zips:
    zpath = os.path.join(CKPT_DIR, zname)
    mb = os.path.getsize(zpath) / 1024 / 1024
    total_ckpt_mb += mb
    epoch_num = zname.replace("adapter_epoch_", "").replace(".zip", "")
    loss = ckpt_callback.epoch_losses.get(int(epoch_num), float("nan")) if epoch_num.isdigit() else float("nan")
    print(f"  {zname:<30}  {mb:6.1f} MB  loss={loss:.4f}")

print(f"\nTotal checkpoint storage: {total_ckpt_mb:.1f} MB  ({len(ckpt_zips)} files)")
print("\nTip: Use the checkpoint with the LOWEST loss for your submission.")

assert "adapter_config.json" in contents, "MISSING adapter_config.json!"
assert "adapter_model.safetensors" in contents, "MISSING adapter_model.safetensors!"

In [ ]:
# ============================================================
# 14. FINAL VERIFICATION — eval-server compliance checklist
# ============================================================
print("=" * 60)
print("  v7.5 TRAINING SUMMARY")
print("=" * 60)

with open(os.path.join(OUTPUT_DIR, "adapter_config.json")) as f:
    final_cfg = json.load(f)

print(f"\n  Model             : Nemotron-3-Nano-30B-A3B (Hybrid)")
print(f"  base_model_name   : {final_cfg.get('base_model_name_or_path')}")
print(f"  LoRA rank (r)     : {final_cfg.get('r')}")
print(f"  LoRA alpha        : {final_cfg.get('lora_alpha')}")
print(f"  Alpha:Rank ratio  : {final_cfg.get('lora_alpha', 0)}:{final_cfg.get('r', 1)}")
print(f"  Target modules    : {final_cfg.get('target_modules')}")
print(f"  Dropout           : {final_cfg.get('lora_dropout')}")
print(f"  Task type         : {final_cfg.get('task_type')}")
print(f"  use_dora          : {final_cfg.get('use_dora', False)}")
print(f"  use_rslora        : {final_cfg.get('use_rslora', False)}")
print(f"\n  Dataset           : {len(hf_dataset)} examples (after filtering)")
print(f"  Categories        : {len(set(hf_dataset['label']))}")
print(f"  Epochs            : {NUM_EPOCHS}")
print(f"  Learning rate     : {LR}")
print(f"  Effective batch   : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Training time     : {elapsed_hrs:.2f} hrs")
print(f"\n  Adapter zip       : {ZIP_PATH} ({zip_mb:.1f} MB)")
print(f"  Adapter files     : {contents}")

targets = final_cfg.get('target_modules', [])
checks = [
    ("base_model = metric/...",    final_cfg.get('base_model_name_or_path') == 'metric/nemotron-3-nano-30b-a3b-bf16'),
    ("peft_type = LORA",            final_cfg.get('peft_type') == 'LORA'),
    ("out_proj IN targets",         'out_proj' in targets),
    ("lm_head IN targets",          'lm_head' in targets),
    ("gate_proj NOT in targets",    'gate_proj' not in targets),
    ("dropout = 0",                 final_cfg.get('lora_dropout', -1) == 0.0),
    ("rank = 32",                   final_cfg.get('r') == 32),
    ("alpha = 64",                  final_cfg.get('lora_alpha') == 64),
    ("use_dora = False/absent",     not final_cfg.get('use_dora', False)),
    ("use_rslora = False/absent",   not final_cfg.get('use_rslora', False)),
]

print(f"\n  Verification checks:")
all_ok = True
for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_ok = False
    print(f"    [{status}] {name}")

if all_ok:
    print(f"\n  All checks passed! Adapter is eval-server compliant.")
    print(f"  -> Download adapter.zip (or best per-epoch checkpoint) from Kaggle Output")
    print(f"  -> Use with nemotron_v75_submission.ipynb (or any plain-LoRA submission notebook)")
else:
    print(f"\n  WARNING: Some checks failed — review before submitting.")

# ---- W&B: log final metrics, finish run, zip logs ----
if WANDB_AVAILABLE:
    wandb.log({
        "final/training_hours": elapsed_hrs,
        "final/dataset_size": len(hf_dataset),
        "final/adapter_zip_mb": zip_mb,
        "final/adapter_file_count": len(contents),
    })
    wandb.finish()

    wandb_dir = os.path.join(WANDB_DIR, "wandb")
    wandb_zip = "/kaggle/working/wandb_logs.zip"
    if os.path.exists(wandb_dir):
        with zipfile.ZipFile(wandb_zip, "w", zipfile.ZIP_DEFLATED) as zf:
            for root, dirs, files in os.walk(wandb_dir):
                for fname in files:
                    fpath = os.path.join(root, fname)
                    arcname = os.path.relpath(fpath, WANDB_DIR)
                    zf.write(fpath, arcname=arcname)
        wandb_zip_mb = os.path.getsize(wandb_zip) / 1024 / 1024
        print(f"\n  W&B logs zipped: {wandb_zip} ({wandb_zip_mb:.1f} MB)")
        print(f"  To view in dashboard locally:")
        print(f"    1. Download wandb_logs.zip from Kaggle Output")
        print(f"    2. Unzip it")
        print(f"    3. Run: wandb sync wandb/offline-run-*")
    else:
        print("\n  W&B directory not found — no logs to zip.")
else:
    print("\n  W&B was disabled — no logs to sync.")